### Movie Recommendation System

This notebook will guide you through building a simple movie recommendation system using basic filtering techniques.

#### 1. Setup and Data Loading

First, we'll load the `movies_metadata.csv` dataset into a pandas DataFrame. This dataset contains information about various movies, which we'll use for our recommendation system.

In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import json
import ast

# Load the dataset
df = pd.read_csv('/content/movies_metadata.csv')

# Display the first 5 rows of the DataFrame
display(df.head())

/tmp/ipykernel_527/2795681905.py:9: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('/content/movies_metadata.csv')


,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,...,release_date,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count
0,False,"{'id': 10194, 'name': 'Toy Story Collection', ...",30000000,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",http://toystory.disney.com/toy-story,862,tt0114709,en,Toy Story,"Led by Woody, Andy's toys live happily in his ...",...,1995-10-30,373554033.0,81.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,NaN,Toy Story,False,7.7,5415.0
1,False,NaN,65000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",NaN,8844,tt0113497,en,Jumanji,When siblings Judy and Peter discover an encha...,...,1995-12-15,262797249.0,104.0,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso...",Released,Roll the dice and unleash the excitement!,Jumanji,False,6.9,2413.0
2,False,"{'id': 119050, 'name': 'Grumpy Old Men Collect...",0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...",NaN,15602,tt0113228,en,Grumpier Old Men,A family wedding reignites the ancient feud be...,...,1995-12-22,0.0,101.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Still Yelling. Still Fighting. Still Ready for...,Grumpier Old Men,False,6.5,92.0
3,False,NaN,16000000,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",NaN,31357,tt0114885,en,Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom...",...,1995-12-22,81452156.0,127.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Friends are the people who let you be yourself...,Waiting to Exhale,False,6.1,34.0
4,False,"{'id': 96871, 'name': 'Father of the Bride Col...",0,"[{'id': 35, 'name': 'Comedy'}]",NaN,11862,tt0113041,en,Father of the Bride Part II,Just when George Banks has recovered from his ...,...,1995-02-10,76578911.0,106.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Just When His World Is Back To Normal... He's ...,Father of the Bride Part II,False,5.7,173.0


To get a better understanding of the data, let's check its shape and column information.

In [2]:
# Display the shape of the DataFrame
print(f"DataFrame shape: {df.shape}")

# Display information about the DataFrame, including data types and non-null values
df.info()

DataFrame shape: (45466, 24)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45466 entries, 0 to 45465
Data columns (total 24 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   adult                  45466 non-null  object 
 1   belongs_to_collection  4494 non-null   object 
 2   budget                 45466 non-null  object 
 3   genres                 45466 non-null  object 
 4   homepage               7782 non-null   object 
 5   id                     45466 non-null  object 
 6   imdb_id                45449 non-null  object 
 7   original_language      45455 non-null  object 
 8   original_title         45466 non-null  object 
 9   overview               44512 non-null  object 
 10  popularity             45461 non-null  object 
 11  poster_path            45080 non-null  object 
 12  production_companies   45463 non-null  object 
 13  production_countries   45463 non-null  object 
 14  release_date           45

#### 2. Data Preprocessing and Cleaning

Based on the `df.info()` output and the `DtypeWarning`, we need to perform several data cleaning steps:

*   **Handle `adult` column**: Convert non-boolean values to `False` and then to boolean type.
*   **Handle `id` column**: Filter out rows where `id` is not a number and convert `id` to numeric type.
*   **Handle `popularity` and `budget` columns**: Convert these columns to numeric types, as the `DtypeWarning` suggests mixed types. Any values that cannot be converted to numbers will be set to `NaN`.
*   **Handle `revenue` column**: Convert this column to numeric type, setting invalid parsing to `NaN`.
*   **Handle `release_date` column**: Convert this column to datetime objects.
*   **Handle `genres` and `production_companies` columns**: These are currently strings representing lists of dictionaries. We'll need to parse them into actual lists of dictionaries and then extract meaningful information.

In [3]:
# Clean 'adult' column
df = df[df['adult'].isin(['True', 'False'])].copy()
df['adult'] = df['adult'].astype(bool)

# Filter out rows where 'id' is not numeric and convert to numeric
non_numeric_ids = df[pd.to_numeric(df['id'], errors='coerce').isna()]['id']
if not non_numeric_ids.empty:
    print(f"Dropping rows with non-numeric IDs: {non_numeric_ids.tolist()}")
df = df[pd.to_numeric(df['id'], errors='coerce').notna()].copy()
df['id'] = pd.to_numeric(df['id'])

# Convert 'budget' and 'popularity' to numeric, setting errors to NaN
df['budget'] = pd.to_numeric(df['budget'], errors='coerce')
df['popularity'] = pd.to_numeric(df['popularity'], errors='coerce')

# Convert 'revenue' to numeric
df['revenue'] = pd.to_numeric(df['revenue'], errors='coerce')

# Convert 'release_date' to datetime
df['release_date'] = pd.to_datetime(df['release_date'], errors='coerce')

# Display the updated info and head to verify changes
print("\nDataFrame info after cleaning:")
df.info()
print("\nDataFrame head after cleaning:")
display(df.head())


DataFrame info after cleaning:
<class 'pandas.core.frame.DataFrame'>
Index: 45463 entries, 0 to 45465
Data columns (total 24 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   adult                  45463 non-null  bool          
 1   belongs_to_collection  4491 non-null   object        
 2   budget                 45463 non-null  int64         
 3   genres                 45463 non-null  object        
 4   homepage               7779 non-null   object        
 5   id                     45463 non-null  int64         
 6   imdb_id                45446 non-null  object        
 7   original_language      45452 non-null  object        
 8   original_title         45463 non-null  object        
 9   overview               44509 non-null  object        
 10  popularity             45460 non-null  float64       
 11  poster_path            45077 non-null  object        
 12  production_companies   45460 non-

,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,...,release_date,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count
0,True,"{'id': 10194, 'name': 'Toy Story Collection', ...",30000000,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",http://toystory.disney.com/toy-story,862,tt0114709,en,Toy Story,"Led by Woody, Andy's toys live happily in his ...",...,1995-10-30,373554033.0,81.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,NaN,Toy Story,False,7.7,5415.0
1,True,NaN,65000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",NaN,8844,tt0113497,en,Jumanji,When siblings Judy and Peter discover an encha...,...,1995-12-15,262797249.0,104.0,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso...",Released,Roll the dice and unleash the excitement!,Jumanji,False,6.9,2413.0
2,True,"{'id': 119050, 'name': 'Grumpy Old Men Collect...",0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...",NaN,15602,tt0113228,en,Grumpier Old Men,A family wedding reignites the ancient feud be...,...,1995-12-22,0.0,101.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Still Yelling. Still Fighting. Still Ready for...,Grumpier Old Men,False,6.5,92.0
3,True,NaN,16000000,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",NaN,31357,tt0114885,en,Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom...",...,1995-12-22,81452156.0,127.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Friends are the people who let you be yourself...,Waiting to Exhale,False,6.1,34.0
4,True,"{'id': 96871, 'name': 'Father of the Bride Col...",0,"[{'id': 35, 'name': 'Comedy'}]",NaN,11862,tt0113041,en,Father of the Bride Part II,Just when George Banks has recovered from his ...,...,1995-02-10,76578911.0,106.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Just When His World Is Back To Normal... He's ...,Father of the Bride Part II,False,5.7,173.0


Now, let's process the `genres` and `production_companies` columns. These columns contain strings that are essentially JSON representations of lists of dictionaries. We'll write a function to safely parse these strings into actual Python lists and extract relevant information (like genre names or company names). This will allow us to use these features for our recommendation system.

In [4]:
# Function to safely parse JSON strings in columns
def parse_json_column(df, column_name):
    def safe_literal_eval(x):
        try:
            return ast.literal_eval(x)
        except (ValueError, SyntaxError):
            return []

    df[column_name] = df[column_name].apply(lambda x: safe_literal_eval(x) if isinstance(x, str) else [])
    return df

# Apply the parsing function to 'genres' and 'production_companies'
df = parse_json_column(df, 'genres')
df = parse_json_column(df, 'production_companies')

# Function to extract names from the parsed lists
def extract_names(list_of_dicts):
    if isinstance(list_of_dicts, list):
        return [d['name'] for d in list_of_dicts if 'name' in d]
    return []

# Extract genre names and production company names
df['genres'] = df['genres'].apply(extract_names)
df['production_companies'] = df['production_companies'].apply(extract_names)

# Display the first few rows with the updated columns
print("\nDataFrame head after parsing 'genres' and 'production_companies':")
display(df[['genres', 'production_companies']].head())


DataFrame head after parsing 'genres' and 'production_companies':


,genres,production_companies
0,"[Animation, Comedy, Family]",[Pixar Animation Studios]
1,"[Adventure, Fantasy, Family]","[TriStar Pictures, Teitler Film, Interscope Co..."
2,"[Romance, Comedy]","[Warner Bros., Lancaster Gate]"
3,"[Comedy, Drama, Romance]",[Twentieth Century Fox Film Corporation]
4,[Comedy],"[Sandollar Productions, Touchstone Pictures]"
